# Settimana 4 — Huygens e il prezzo equo: il valore atteso e la linearità

Questo è il notebook della Settimana 4 del corso di Probabilità e Statistica. Contiene gli script Python della [dispensa della settimana](https://qwertyjacob.github.io/PSI-VA-2627/settimana_04/), nello stesso ordine in cui compaiono: ogni script è una cella di codice, preceduta dal titolo della sezione in cui si trova. Le spiegazioni restano nella dispensa.

Per eseguire una cella, premi il pulsante ▶ alla sua sinistra. Gli script sono indipendenti l'uno dall'altro, quindi puoi eseguirli in qualsiasi ordine e modificarli liberamente. Colab non salva le modifiche nel notebook originale: se vuoi conservarle, scegli *File → Salva una copia in Drive*.

## 6. Laboratorio: calcolare il valore atteso passo per passo

[Leggi la spiegazione nella dispensa](https://qwertyjacob.github.io/PSI-VA-2627/settimana_04/#lab-pmf)

In [ ]:
from fractions import Fraction

# Ogni coppia contiene (pagamento in euro, probabilità).
pagamenti = [(0, Fraction(3, 4)), (8, Fraction(1, 4))]


def media(distribuzione):
    # Partiamo da zero: non abbiamo ancora letto nessuna coppia.
    totale_probabilita = Fraction(0, 1)
    valore_atteso = Fraction(0, 1)

    for valore, probabilita in distribuzione:
        # assert ferma l'esecuzione se la condizione è falsa.
        assert probabilita >= 0, "Una probabilità non può essere negativa"

        # Aggiungiamo la probabilità corrente a quelle già lette.
        totale_probabilita = totale_probabilita + probabilita

        # Il contributo alla media è il valore moltiplicato per il suo peso.
        contributo = valore * probabilita
        valore_atteso = valore_atteso + contributo

    # Finito il ciclo, controlliamo di avere tutta la probabilità.
    assert totale_probabilita == 1, "Le probabilità devono sommare a 1"

    # Restituiamo a chi ha chiamato la funzione la media calcolata.
    return valore_atteso


# La funzione riceve la lista pagamenti; print mostra il risultato.
print("Pagamento medio:", media(pagamenti), "euro")

# Usiamo la stessa funzione per il numero di teste in due lanci indipendenti
# di una moneta onesta: cambiano i dati, ma il procedimento resta uguale.
teste = [(0, Fraction(1, 4)),
         (1, Fraction(1, 2)),
         (2, Fraction(1, 4))]
print("Numero medio di teste:", media(teste))

## 11. Laboratorio: contare coppie e contare esperimenti

[Leggi la spiegazione nella dispensa](https://qwertyjacob.github.io/PSI-VA-2627/settimana_04/#lab-hash)

In [ ]:
from math import comb, prod
from random import Random

generatore = Random(42)
n, m, prove = 23, 365, 10000
assert n >= 1 and m >= 1 and prove >= 1
totale_coppie = 0
con_collisione = 0

for _ in range(prove):
    # Ogni chiave riceve un indice uniforme, indipendente dagli altri.
    indici = [generatore.randrange(m) for _ in range(n)]
    coppie = sum(indici[i] == indici[j]
                 for i in range(n) for j in range(i + 1, n))
    totale_coppie += coppie
    con_collisione += coppie > 0

# Se n > m, almeno una coppia deve condividere una casella.
p_almeno_una = 1.0 if n > m else 1 - prod(
    (m - i) / m for i in range(n)
)
print(f"Media teorica delle coppie: {comb(n, 2) / m:.4f}")
print(f"Media simulata delle coppie: {totale_coppie / prove:.4f}")
print(f"Probabilità teorica di almeno una: {p_almeno_una:.4f}")
print(f"Frazione simulata con collisione: {con_collisione / prove:.4f}")

## 15. Laboratorio: indipendenza e dipendenza a confronto

[Leggi la spiegazione nella dispensa](https://qwertyjacob.github.io/PSI-VA-2627/settimana_04/#lab-binomiale)

In [ ]:
from math import comb

import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng(42)
n, p, prove = 4, 0.2, 20000
# Nel primo modello ogni prova riceve una decisione distinta.
indipendenti = (rng.random((prove, n)) < p).sum(axis=1)
# Nel secondo un solo bit comune accende tutti i fallimenti.
dipendenti = n * (rng.random(prove) < p)
k = np.arange(n + 1)
pmf = np.array([comb(n, int(j)) * p**j * (1 - p)**(n - j)
                for j in k])
assert np.isclose(pmf.sum(), 1)
assert np.isclose(np.dot(k, pmf), n * p)

print(f"Media teorica di entrambi: {n * p:.3f}")
print(f"Media simulata, indipendenti: {indipendenti.mean():.3f}")
print(f"Media simulata, dipendenti: {dipendenti.mean():.3f}")
plt.bar(k - 0.2, pmf, width=0.4, label="Distribuzione binomiale teorica")
plt.bar(k + 0.2, np.bincount(dipendenti, minlength=n + 1) / prove,
        width=0.4, label="Guasto comune simulato")
plt.xticks(k)
plt.xlabel("Numero di fallimenti")
plt.ylabel("Probabilità / frequenza relativa")
plt.legend()
plt.tight_layout()
plt.show()